## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 2. Import and Inspect the Dataset

In [2]:
df = pd.read_csv('titanic.csv')
print("Shape:", df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
df.describe(include='all')

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [5]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

,missing_count,missing_pct
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


In [6]:
# Duplicate records
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated PassengerId values:", df['PassengerId'].duplicated().sum())

Fully duplicated rows: 0
Duplicated PassengerId values: 0


In [7]:
# Data types
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

## 4. Handle Missing Values

In [8]:
df_clean = df.copy()

# Age: impute with the median, grouped by Pclass and Sex (more accurate than a single global median)
df_clean['Age'] = df_clean.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda s: s.fillna(s.median())
)

# Embarked: impute the 2 missing rows with the mode (most frequent port)
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Cabin: too sparse (77% missing) to impute meaningfully.
# Instead of dropping the column outright, we preserve the signal by flagging
# whether cabin info is known, then drop the raw Cabin column.
df_clean['CabinKnown'] = df_clean['Cabin'].notna().astype(int)
df_clean = df_clean.drop(columns=['Cabin'])

print(df_clean.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
CabinKnown     0
dtype: int64


## 5. Detect and Remove Duplicate Records

In [9]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Rows before: {before}, after: {after}, duplicates removed: {before - after}")

Rows before: 891, after: 891, duplicates removed: 0


## 6. Verify and Correct Data Types

In [10]:
# Age has no more missing values, so it can safely become an integer (years)
df_clean['Age'] = df_clean['Age'].round().astype(int)

# Categorical columns are better stored as category dtype for memory/analysis efficiency
for col in ['Pclass', 'Sex', 'Embarked', 'Survived']:
    df_clean[col] = df_clean[col].astype('category')

df_clean.dtypes

PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex            category
Age               int64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked       category
CabinKnown        int64
dtype: object

## 7. Rename Columns for Readability

In [11]:
df_clean = df_clean.rename(columns={
    'PassengerId': 'passenger_id',
    'Survived': 'survived',
    'Pclass': 'pclass',
    'Name': 'name',
    'Sex': 'sex',
    'Age': 'age',
    'SibSp': 'siblings_spouses_aboard',
    'Parch': 'parents_children_aboard',
    'Ticket': 'ticket',
    'Fare': 'fare',
    'Embarked': 'embarked',
    'CabinKnown': 'cabin_known'
})
df_clean.head()

,passenger_id,survived,pclass,name,sex,age,siblings_spouses_aboard,parents_children_aboard,ticket,fare,embarked,cabin_known
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,S,0


## 8. Final Check

In [12]:
print("Final shape:", df_clean.shape)
print("\nRemaining missing values:\n", df_clean.isnull().sum().sum())
print("\nDtypes:\n", df_clean.dtypes)
df_clean.describe(include='all')

Final shape: (891, 12)

Remaining missing values:
 0

Dtypes:
 passenger_id                  int64
survived                   category
pclass                     category
name                            str
sex                        category
age                           int64
siblings_spouses_aboard       int64
parents_children_aboard       int64
ticket                          str
fare                        float64
embarked                   category
cabin_known                   int64
dtype: object


,passenger_id,survived,pclass,name,sex,age,siblings_spouses_aboard,parents_children_aboard,ticket,fare,embarked,cabin_known
count,891.000000,891.0,891.0,891,891,891.000000,891.000000,891.000000,891,891.000000,891,891.000000
unique,NaN,2.0,3.0,891,2,NaN,NaN,NaN,681,NaN,3,NaN
top,NaN,0.0,3.0,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,S,NaN
freq,NaN,549.0,491.0,1,577,NaN,NaN,NaN,7,NaN,646,NaN
mean,446.000000,NaN,NaN,NaN,NaN,29.131313,0.523008,0.381594,NaN,32.204208,NaN,0.228956
std,257.353842,NaN,NaN,NaN,NaN,13.289416,1.102743,0.806057,NaN,49.693429,NaN,0.420397
min,1.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,0.000000,NaN,0.000000
25%,223.500000,NaN,NaN,NaN,NaN,22.000000,0.000000,0.000000,NaN,7.910400,NaN,0.000000
50%,446.000000,NaN,NaN,NaN,NaN,26.000000,0.000000,0.000000,NaN,14.454200,NaN,0.000000
75%,668.500000,NaN,NaN,NaN,NaN,36.000000,1.000000,0.000000,NaN,31.000000,NaN,0.000000


## 9. Save the Cleaned Dataset

In [13]:
df_clean.to_csv('cleaned_titanic.csv', index=False)
print("Saved cleaned_titanic.csv with shape:", df_clean.shape)

Saved cleaned_titanic.csv with shape: (891, 12)
